In [6]:
import os
import sys
import time

import joblib
import pandas as pd
import numpy as np
from tslearn.metrics import cdist_dtw

sys.path.insert(0, '../../..') # Load Parent directory to import utils
from utils.constants import Configs
from utils.helper_functions import normalize_series # calcula a distância entre séries temporais usando DTW

### Load Data From cache

In [7]:
cache_path = Configs.SMART_RETAIL_FILTERED_DATA_BIMANUAL
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")
df = pd.read_csv(cache_path)

### Delete Subject 13

In [8]:
# df.drop(df[df['sub']== 13].index, inplace=True)
# df[df['sub']==13].head()

### Calculate  constrainted DTW Matrix for all movements
I will make a simple mean to fuse both distance matrices
$$\text{dtw\_matrix\_final} = 0.5 \times \text{dtw\_matrix\_left} + 0.5 \times \text{dtw\_matrix\_right}$$

Global Warping constraint Sakoe Chiba band is tunned  in each phase

In [9]:
group_left = ['x_L', 'y_L', 'z_L']
group_right = ['x_R', 'y_R', 'z_R']
sample_size = None

# tslearn naturally computes Dependent DTW (DTWD) when passing 3D arrays
# We will test different window constraints
window_percentages = [0.05, 0.10, 0.15, 0.20]

In [10]:
times = []

for movement in ["Pick", "Park", "Place"]:
    print(f"\n--- Tuning Constrained Bimanual DTW (Mean Fusion) for {movement} ---")

    label_df = df[df["movLabel"] == movement]

    # Extrai Esquerda
    normalized_data_left = normalize_series(
            df_input=label_df,
            movLabel_filter=movement,
            useScaler=False,
            group_array=group_left,
            sample_size=sample_size
    )  # shape (n_samples, T, 9)

    # Extrai Direita
    normalized_data_right = normalize_series(
            df_input=label_df,
            movLabel_filter=movement,
            useScaler=False,
            group_array=group_right,
            sample_size=sample_size
    )  # shape (N_samples, T, 9)

    if normalized_data_left.shape[0] != normalized_data_right.shape[0]:
        raise ValueError("Error: Número de amostras esquerda/direita não bate certo!")

    seq_length = normalized_data_left.shape[1]
    
    best_score = -1
    best_radius = None
    best_dtw_matrix = None
    best_time = 0.0

    for w_pct in window_percentages:
        radius = max(1, int(seq_length * w_pct))
        
        start_time = time.perf_counter()
        
        # Matriz Esquerda com Sakoe-Chiba
        dtw_matrix_left = cdist_dtw(
            normalized_data_left, 
            global_constraint="sakoe_chiba", 
            sakoe_chiba_radius=radius, 
            n_jobs=-1
        ) 
        
        # Matriz Direita com Sakoe-Chiba
        dtw_matrix_right = cdist_dtw(
            normalized_data_right, 
            global_constraint="sakoe_chiba", 
            sakoe_chiba_radius=radius, 
            n_jobs=-1
        ) 

        # Fusing matrices (Mean 50/50)
        dtw_matrix_fused = (0.5 * dtw_matrix_left) + (0.5 * dtw_matrix_right)

        elapsed_time = time.perf_counter() - start_time
        
        # Avaliar a capacidade de separação da matriz fundida
        upper_tri = dtw_matrix_fused[np.triu_indices_from(dtw_matrix_fused, k=1)]
        separation_score = np.var(upper_tri)
        
        print(f"  -> Radius: {radius} ({w_pct*100:.0f}%) | Time: {elapsed_time:.2f}s | Separation Score: {separation_score:.2f}")
        
        if separation_score > best_score:
            best_score = separation_score
            best_radius = radius
            best_dtw_matrix = dtw_matrix_fused
            best_time = elapsed_time

    times.append(best_time)
    print(f"[*] Selected optimal Sakoe-Chiba radius for {movement}: {best_radius}")
    print(f"[*] Final Representation Time (Left + Right + Mean Fusion): {best_time:.4f}s")

    # Export best matrix
    cache_name = str(Configs.mean_dtw_matrix_path(movement))
    joblib.dump(best_dtw_matrix, cache_name)
    print(f"[*] Saved optimal fused matrix to {cache_name}")


--- Tuning Constrained Bimanual DTW (Mean Fusion) for Pick ---


Processing time series: 100%|██████████| 40/40 [00:00<00:00, 1944.42it/s]


Shape of first series: (106, 3)
Shape of formatted series: (40, 226, 3)


Processing time series: 100%|██████████| 40/40 [00:00<00:00, 5233.07it/s]

Shape of first series: (106, 3)
Shape of formatted series: (40, 226, 3)


  -> Radius: 11 (5%) | Time: 4.99s | Separation Score: 1780333.75
  -> Radius: 22 (10%) | Time: 0.91s | Separation Score: 1783220.11
  -> Radius: 33 (15%) | Time: 0.85s | Separation Score: 1783513.74
  -> Radius: 45 (20%) | Time: 0.89s | Separation Score: 1783607.99
[*] Selected optimal Sakoe-Chiba radius for Pick: 45
[*] Final Representation Time (Left + Right + Mean Fusion): 0.8872s
[*] Saved optimal fused matrix to /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/SmartRetail/data/processed/dtw_mean_distance_matrix_Pick.csv

--- Tuning Constrained Bimanual DTW (Mean Fusion) for Park ---


... (6 outputs left out)

Shape of first series: (121, 3)
Shape of formatted series: (40, 251, 3)


Processing time series: 100%|██████████| 40/40 [00:00<00:00, 4919.28it/s]

Shape of first series: (121, 3)
Shape of formatted series: (40, 251, 3)


  -> Radius: 12 (5%) | Time: 0.87s | Separation Score: 1916192.79
  -> Radius: 25 (10%) | Time: 1.00s | Separation Score: 1922067.94
  -> Radius: 37 (15%) | Time: 1.07s | Separation Score: 1917065.53
  -> Radius: 50 (20%) | Time: 0.98s | Separation Score: 1916803.42
[*] Selected optimal Sakoe-Chiba radius for Place: 25
[*] Final Representation Time (Left + Right + Mean Fusion): 0.9984s
[*] Saved optimal fused matrix to /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/SmartRetail/data/processed/dtw_mean_distance_matrix_Place.csv


In [11]:
print("\nFinal times per phase [Pick, Park, Place]:", [round(t, 2) for t in times])


Final times per phase [Pick, Park, Place]: [0.89, 0.1, 1.0]
